# Debug: Two-Stage Feature Importance Parsing

**Problem**: The `TS XGB→Ada` (`two_stage_xgb_ada`) model shows *"No feature importance data available"* in `fig_feature_importance_2.png`, even though the model trains successfully with F1 = 0.600. The `Ordinal RF` on the same page renders fine.

**Root cause hypothesis**: The two-stage pipeline stores `weights_json` in a nested format:
```json
{"stage_1": {"feat_a": 0.3, ...}, "stage_2": {"feat_b": 0.1, ...}}
```
The visualization code has two paths to load this:
1. **Fluent API** (`rs.features().top(top_n)`) — doesn't handle nested format
2. **SQL fallback** (`_parse_multistage_weights`) — *should* handle it but never executes

This notebook walks through each path to find where it breaks.

In [2]:
from pathlib import Path
import sys
import os

# Automatically find repo root by looking for .git
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

os.chdir(ROOT)

print(f"Current root: {ROOT}")

Current root: c:\Users\rjbel\Python\Notebooks\Mapua\Thesis


---
## 1. Load the database and locate the experiment

In [3]:
import sys, json, warnings, sqlite3
import numpy as np

# Adjust this path to your project root if needed
# sys.path.insert(0, "/path/to/project")

from src.modules.machine_learning.utils.io import ResultsAnalyzer

RESULTS_DIR = "data/training_results/"
MODEL_SLUG  = "two_stage_xgb_ada"

ra = ResultsAnalyzer(RESULTS_DIR)
rs = ra.model(MODEL_SLUG).top(1)

print(f"Result set empty? {rs.df.empty}")
if not rs.df.empty:
    row = rs.df.iloc[0]
    exp_id = int(row["id"])
    print(f"Experiment ID: {exp_id}")
    print(f"Strategy: {row.get('strategy_label', row.get('balance_strategy', '?'))}")
    print(f"F1: {row.get('enhanced_f1_macro', row.get('f1_macro', '?'))}")

Result set empty? False
Experiment ID: 814
Strategy: borderline_smote
F1: 0.6002784350998337


---
## 2. Test Path 1 — Fluent API (`rs.features().top()`)

This is the first path tried by `_load_feature_data`. It returns 20 features but all with weight 0.0 — it can't parse the nested stage format.

In [4]:
try:
    feat_result = rs.features().top(20)
    feat_dict = feat_result.as_dict()
    print(f"Fluent API returned keys: {list(feat_dict.keys())}")
    for k, v in feat_dict.items():
        print(f"  {k}: {len(v)} features")
        if v:
            print(f"    first 3: {v[:3]}")
except Exception as e:
    print(f"Fluent API raised: {type(e).__name__}: {e}")

Fluent API returned keys: ['two_stage_xgb_ada']
  two_stage_xgb_ada: 20 features
    first 3: [('credit_sale_amount', 0.0), ('dtp_1', 0.0), ('dtp_2', 0.0)]


---
## 3. ROOT CAUSE — zero-weight guard blocks the SQL fallback

The fluent API returns 20 features with `weight = 0.0`. The visualization code filters zeros with `w > 0`, leaving nothing. But the SQL fallback condition checks `if not raw_pairs:` — which is `False` because the list has 20 items. So the fallback that *can* parse multi-stage weights **never runs**.

In [5]:
raw_pairs = []
try:
    feat_result = rs.features().top(20)
    feat_dict = feat_result.as_dict()
    if MODEL_SLUG in feat_dict:
        raw_pairs = feat_dict[MODEL_SLUG]
    else:
        raw_pairs = next(iter(feat_dict.values()), [])
except Exception:
    pass

print(f"raw_pairs count:      {len(raw_pairs)}")
print(f"Non-zero weights:     {sum(1 for _, w in raw_pairs if w > 0)}")
print(f"'not raw_pairs':      {not raw_pairs}   ← current fallback condition")
print(f"After zero-filter:    {len([(f, w) for f, w in raw_pairs if w > 0])} features")
print()
print("BUG: The fluent API returns features with 0.0 weights (it can't parse")
print("the nested stage format), so raw_pairs is truthy but useless.")
print("The SQL fallback is skipped because 'not raw_pairs' is False.")
print()
print("FIX in _load_feature_data — change:")
print("    if not raw_pairs:")
print("to:")
print("    if not raw_pairs or all(w == 0 for _, w in raw_pairs):")

raw_pairs count:      20
Non-zero weights:     0
'not raw_pairs':      False   ← current fallback condition
After zero-filter:    0 features

BUG: The fluent API returns features with 0.0 weights (it can't parse
the nested stage format), so raw_pairs is truthy but useless.
The SQL fallback is skipped because 'not raw_pairs' is False.

FIX in _load_feature_data — change:
    if not raw_pairs:
to:
    if not raw_pairs or all(w == 0 for _, w in raw_pairs):


---
## 4. Get a DB connection

`ResultsRepository` has no persistent connection attribute — it uses a `_connect()` method and exposes `db_path`. We open the DB directly.

In [6]:
repo = ra.repo
conn = None

# Strategy 1: call repo._connect() if it returns a connection
if hasattr(repo, "_connect"):
    try:
        conn = repo._connect()
        if not isinstance(conn, sqlite3.Connection):
            print(f"repo._connect() returned {type(conn).__name__}, not a Connection")
            conn = None
        else:
            print(f"Got connection via repo._connect()")
    except Exception as e:
        print(f"repo._connect() raised: {e}")

# Strategy 2: open db_path directly
if conn is None and hasattr(repo, "db_path"):
    db_path = repo.db_path
    print(f"Opening DB directly from repo.db_path = '{db_path}'")
    conn = sqlite3.connect(db_path)

if conn:
    # Quick sanity check
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    print(f"Connection OK — tables: {[t[0] for t in tables]}")
else:
    print("Could not get a DB connection.")
    print(f"repo attributes: {[a for a in dir(repo) if not a.startswith('__')]}")

repo._connect() returned _GeneratorContextManager, not a Connection
Opening DB directly from repo.db_path = 'data/training_results/2026_05_09_02\results.db'
Connection OK — tables: ['cache_registry', 'charts', 'class_mappings', 'experiments', 'features', 'metadata', 'metrics', 'schema_version', 'sqlite_sequence', 'survival_results']


---
## 5. Inspect all feature rows for this experiment

Check what phases exist and whether `weights_json` is populated.

In [7]:
if conn is None:
    print("Skipped — no DB connection found in cell 4.")
else:
    cursor = conn.execute(
        "SELECT id, phase, "
        "       LENGTH(weights_json) as wj_len, "
        "       SUBSTR(weights_json, 1, 200) as wj_preview "
        "FROM features WHERE experiment_id = ? ORDER BY id",
        (exp_id,),
    )
    rows = cursor.fetchall()
    print(f"Total feature rows for experiment {exp_id}: {len(rows)}")
    for r in rows:
        print(f"\n  id={r[0]}, phase='{r[1]}', wj_len={r[2]}")
        print(f"    preview: {r[3]}")

Total feature rows for experiment 814: 2

  id=1627, phase='baseline', wj_len=1046
    preview: {"stage_1": {"dtp_2": 0.017819, "dtp_3": 0.022961, "dtp_avg": 0.0181, "dtp_wavg": 0.015679, "dtp_2_trend": 0.033269, "dtp_3_trend": 0.024841, "opening_balance": 0.145093, "plan_type_Plan - A": 0.01516

  id=1628, phase='enhanced', wj_len=1821
    preview: {"stage_1": {"credit_sale_amount": 0.013511, "dtp_2": 0.009967, "dtp_3": 0.013805, "dtp_2_trend": 0.024958, "dtp_3_trend": 0.01706, "days_since_last_payment": 0.010852, "opening_balance": 0.104095, "p


---
## 6. Parse the raw `weights_json` directly

Load each feature row's `weights_json` and run the parser to confirm it *can* produce results when actually reached.

In [8]:
def _parse_multistage_weights(weights_raw, top_n=20):
    """Exact copy from the visualization code for debugging."""
    if not weights_raw:
        return []

    try:
        weights = json.loads(weights_raw) if isinstance(weights_raw, str) else weights_raw
    except (json.JSONDecodeError, TypeError) as exc:
        warnings.warn(f"Could not parse weights_json: {exc}")
        return []

    if not isinstance(weights, dict):
        return []

    stage_keys = [k for k in weights if k.startswith("stage_") and isinstance(weights[k], dict)]

    if stage_keys:
        merged = {}
        for sk in stage_keys:
            stage_dict = weights[sk]
            for feat, score in stage_dict.items():
                try:
                    score_f = float(score)
                except (TypeError, ValueError):
                    continue
                if feat in merged:
                    merged[feat] = max(merged[feat], score_f)
                else:
                    merged[feat] = score_f
        weights = merged

    if isinstance(weights, dict):
        pairs = []
        for feat, score in weights.items():
            try:
                pairs.append((feat, float(score)))
            except (TypeError, ValueError):
                continue
        pairs.sort(key=lambda x: x[1], reverse=True)
        return pairs[:top_n]

    return []


if conn is None:
    print("Skipped — no DB connection.")
else:
    cursor = conn.execute(
        "SELECT phase, weights_json FROM features WHERE experiment_id = ? ORDER BY id",
        (exp_id,),
    )
    all_feat_rows = cursor.fetchall()

    for phase, wj in all_feat_rows:
        print(f"\n{'='*60}")
        print(f"Phase: '{phase}'")
        print(f"weights_json type: {type(wj)}, truthy: {bool(wj)}")

        if not wj:
            print("  → EMPTY / NULL")
            continue

        # Show structure
        try:
            parsed = json.loads(wj) if isinstance(wj, str) else wj
            print(f"Parsed type: {type(parsed)}")

            if isinstance(parsed, dict):
                print(f"Top-level keys: {list(parsed.keys())[:10]}")
                stage_keys = [k for k in parsed if k.startswith("stage_") and isinstance(parsed[k], dict)]
                print(f"Stage keys found: {stage_keys}")

                for sk in stage_keys:
                    stage_dict = parsed[sk]
                    print(f"  {sk}: {len(stage_dict)} features")
                    sorted_feats = sorted(stage_dict.items(), key=lambda x: float(x[1]), reverse=True)
                    for feat, score in sorted_feats[:5]:
                        print(f"    {feat}: {score}")

                non_stage = [k for k in parsed if k not in stage_keys]
                if non_stage:
                    print(f"  Non-stage keys: {non_stage[:5]}")
        except Exception as e:
            print(f"  Parse error: {e}")

        # Run the parser
        result = _parse_multistage_weights(wj, top_n=20)
        print(f"\n  _parse_multistage_weights → {len(result)} features")
        if result:
            for feat, score in result[:5]:
                print(f"    {feat}: {score:.6f}")
            print("  ✓ Parser works — the SQL fallback just needs to be reached!")
        else:
            print("  ✗ Parser returned empty")


Phase: 'baseline'
weights_json type: <class 'str'>, truthy: True
Parsed type: <class 'dict'>
Top-level keys: ['stage_1', 'stage_2']
Stage keys found: ['stage_1', 'stage_2']
  stage_1: 14 features
    opening_balance_flag: 0.496091
    opening_balance: 0.145093
    payment_ratio: 0.078297
    plan_type_risk_score: 0.039691
    due_month: 0.03806
  stage_2: 27 features
    dtp_avg: 0.477779
    payment_ratio: 0.19593
    dtp_wavg: 0.146168
    dtp_1: 0.054354
    opening_balance_flag: 0.04112

  _parse_multistage_weights → 20 features
    opening_balance_flag: 0.496091
    dtp_avg: 0.477779
    payment_ratio: 0.195930
    dtp_wavg: 0.146168
    opening_balance: 0.145093
  ✓ Parser works — the SQL fallback just needs to be reached!

Phase: 'enhanced'
weights_json type: <class 'str'>, truthy: True
Parsed type: <class 'dict'>
Top-level keys: ['stage_1', 'stage_2']
Stage keys found: ['stage_1', 'stage_2']
  stage_1: 24 features
    opening_balance_flag: 0.348183
    opening_balance: 0.10409

---
## 7. Replicate `_load_feature_data` — current (buggy) vs fixed

Side-by-side comparison showing the current logic fails and the fixed logic succeeds.

In [9]:
model_slug = MODEL_SLUG

rs = ra.model(model_slug).top(1)
row = rs.df.iloc[0]
exp_id = int(row["id"])

# ── Fluent API (same for both) ────────────────────────────────────────────
raw_pairs = []
try:
    feat_result = rs.features().top(20)
    feat_dict = feat_result.as_dict()
    if model_slug in feat_dict:
        raw_pairs = feat_dict[model_slug]
    else:
        raw_pairs = next(iter(feat_dict.values()), [])
except Exception:
    pass

print(f"Fluent API: {len(raw_pairs)} pairs, "
      f"non-zero: {sum(1 for _, w in raw_pairs if w > 0)}")

# ── CURRENT (buggy) condition ─────────────────────────────────────────────
print(f"\n--- CURRENT condition: 'not raw_pairs' = {not raw_pairs} ---")
if not raw_pairs:
    print("  Would enter SQL fallback")
else:
    print("  ✗ Skips SQL fallback — all zeros pass through to zero-filter → empty")

current_result = [(f, w) for f, w in raw_pairs if w > 0]
print(f"  Final features: {len(current_result)}  ← 'No data available'")

# ── FIXED condition ───────────────────────────────────────────────────────
has_signal = raw_pairs and any(w > 0 for _, w in raw_pairs)
print(f"\n--- FIXED condition: 'not has_signal' = {not has_signal} ---")

if not has_signal and conn is not None:
    print("  ✓ Entering SQL fallback...")
    cursor = conn.execute(
        "SELECT phase, weights_json FROM features "
        "WHERE experiment_id = ? AND weights_json IS NOT NULL "
        "AND weights_json != '' AND LENGTH(weights_json) > 2 "
        "ORDER BY CASE phase WHEN 'enhanced' THEN 0 ELSE 1 END, id DESC",
        (exp_id,),
    )
    for phase, wj in cursor.fetchall():
        fixed_pairs = _parse_multistage_weights(wj, top_n=20)
        if fixed_pairs:
            fixed_pairs = [(f, w) for f, w in fixed_pairs if w > 0]
            if fixed_pairs:
                max_w = max(w for _, w in fixed_pairs)
                fixed_pairs = [(f, w / max_w) for f, w in fixed_pairs]
            print(f"  phase='{phase}': {len(fixed_pairs)} features after normalize")
            for f, w in fixed_pairs[:10]:
                print(f"    {f}: {w:.4f}")
            print(f"  ✓ Chart would render with {len(fixed_pairs)} bars!")
            break
elif conn is None:
    print("  Skipped — no DB connection (update cell 4)")

Fluent API: 20 pairs, non-zero: 0

--- CURRENT condition: 'not raw_pairs' = False ---
  ✗ Skips SQL fallback — all zeros pass through to zero-filter → empty
  Final features: 0  ← 'No data available'

--- FIXED condition: 'not has_signal' = True ---
  ✓ Entering SQL fallback...
  phase='enhanced': 20 features after normalize
    dtp_avg: 1.0000
    opening_balance_flag: 0.7288
    payment_ratio: 0.4101
    dtp_wavg: 0.3059
    opening_balance: 0.2179
    expected_survival_time: 0.1512
    surv_prob_58: 0.1322
    dtp_1: 0.1138
    surv_prob_306: 0.0728
    plan_type_risk_score: 0.0619
  ✓ Chart would render with 20 bars!


---
## 8. Synthetic tests — verify `_parse_multistage_weights` logic

In [10]:
# Test 1: Standard multi-stage format
test1 = json.dumps({
    "stage_1": {"opening_balance": 0.35, "payment_ratio": 0.28, "dtp_avg": 0.10},
    "stage_2": {"dtp_avg": 0.40, "dtp_1": 0.25, "opening_balance": 0.05},
})
r1 = _parse_multistage_weights(test1)
print("Test 1 (standard multi-stage):")
for f, w in r1:
    print(f"  {f}: {w}")
assert len(r1) == 4, f"Expected 4 merged features, got {len(r1)}"
assert dict(r1)["dtp_avg"] == 0.40, "Max-merge failed for dtp_avg"
assert dict(r1)["opening_balance"] == 0.35, "Max-merge failed for opening_balance"
print("  ✓ All assertions passed\n")

# Test 2: Flat dict (non-two-stage model)
test2 = json.dumps({"feat_a": 0.5, "feat_b": 0.3})
r2 = _parse_multistage_weights(test2)
print("Test 2 (flat dict):")
for f, w in r2:
    print(f"  {f}: {w}")
assert len(r2) == 2
print("  ✓ Passed\n")

# Test 3: Empty stage dicts
test3 = json.dumps({"stage_1": {}, "stage_2": {}})
r3 = _parse_multistage_weights(test3)
print(f"Test 3 (empty stages): {r3}")
assert r3 == []
print("  ✓ Passed\n")

# Test 4: None / empty
assert _parse_multistage_weights(None) == []
assert _parse_multistage_weights("") == []
print("Test 4 (None/empty): ✓ Passed\n")

# Test 5: Mixed dict with metadata
test5 = json.dumps({
    "stage_1": {"feat_a": 0.5},
    "stage_2": {"feat_b": 0.3},
    "method": "two_stage",
    "n_features": 20,
})
r5 = _parse_multistage_weights(test5)
print(f"Test 5 (mixed with metadata): {len(r5)} features")
for f, w in r5:
    print(f"  {f}: {w}")
assert len(r5) == 2, "Metadata keys should be excluded"
print("  ✓ Passed")

Test 1 (standard multi-stage):
  dtp_avg: 0.4
  opening_balance: 0.35
  payment_ratio: 0.28
  dtp_1: 0.25
  ✓ All assertions passed

Test 2 (flat dict):
  feat_a: 0.5
  feat_b: 0.3
  ✓ Passed

Test 3 (empty stages): []
  ✓ Passed

Test 4 (None/empty): ✓ Passed

Test 5 (mixed with metadata): 2 features
  feat_a: 0.5
  feat_b: 0.3
  ✓ Passed


---
## 9. Check the DB schema and distinct phases

In [11]:
if conn is None:
    print("Skipped — no DB connection.")
else:
    # Table schema
    cursor = conn.execute("PRAGMA table_info(features)")
    cols = cursor.fetchall()
    print("features table schema:")
    for c in cols:
        print(f"  {c[1]:20s} {c[2]:10s} {'NOT NULL' if c[3] else 'nullable':>10s}")

    # Distinct phases for this experiment
    print(f"\nDistinct phases for experiment {exp_id}:")
    cursor = conn.execute(
        "SELECT DISTINCT phase FROM features WHERE experiment_id = ?",
        (exp_id,),
    )
    for r in cursor.fetchall():
        print(f"  '{r[0]}'")

features table schema:
  id                   INTEGER      nullable
  experiment_id        INTEGER      NOT NULL
  phase                TEXT         NOT NULL
  feature_method       TEXT         nullable
  feature_parameters   TEXT         nullable
  features_json        TEXT         nullable
  weights_json         TEXT         nullable

Distinct phases for experiment 814:
  'baseline'
  'enhanced'


---
## 10. Inspect how results are saved — call order in `fit()`

Verify that `_write_per_stage_weights()` is called during `fit()` and that the result-saving code sees the nested dict.

In [12]:
import inspect

try:
    from src.modules.machine_learning.models.two_stage_classifier import TwoStagePipeline
    print("TwoStagePipeline.fit() source:")
    print(inspect.getsource(TwoStagePipeline.fit))
except Exception as e:
    print(f"Could not inspect TwoStagePipeline.fit: {e}")
    print("\nExpected call order (from uploaded source):")
    print("  1. self.model.fit(X_train, y_train)")
    print("  2. self._set_features(method_text='none')     ← sets flat weights")
    print("  3. self._write_per_stage_weights()             ← overwrites with nested")
    print("")
    print("If the DB save happens between steps 2 and 3, the flat (zero) weights")
    print("are persisted instead of the nested stage weights.")

TwoStagePipeline.fit() source:
    def fit(self, use_feature_selection=False, threshold="median"):
        """
        Train the two-stage model, optionally applying independent per-stage
        feature selection.

        When ``use_feature_selection=True`` the procedure is:

        1. Fit the full ``TwoStageClassifier`` on all features so both
           stages have trained estimators from which importances can be read.
        2. Call ``_build_stage_masks`` to derive ``mask1`` and ``mask2`` from
           each stage's own importances independently.
        3. Call ``_log_feature_selection`` to record the union mask and
           feature metadata to ``self.features`` via ``_set_features``.
        4. Call ``fit_with_masks`` to retrain each stage on its own feature
           subset.  ``predict_proba`` then applies the stored masks automatically.
        5. Call ``_write_per_stage_weights`` to overwrite the flat weights with
           a nested ``{"stage_1": …, "stage_2": …}`` dic

---
## 11. Also check: how does the visualization code get the connection?

The SQL fallback in `_load_feature_data` uses `ra.repo._conn` which doesn't exist. This is a **secondary bug** — even with the fixed fallback condition, the SQL path would crash. It needs to use `ra.repo._connect()` or `sqlite3.connect(ra.repo.db_path)` instead.

In [13]:
# Confirm the visualization code's connection access pattern is broken
print("Checking ra.repo._conn ...")
try:
    _ = ra.repo._conn
    print("  ✓ ra.repo._conn exists (unexpected!)")
except AttributeError:
    print("  ✗ ra.repo._conn does NOT exist — secondary bug confirmed")

print("\nChecking ra.repo._connect() ...")
try:
    test_conn = ra.repo._connect()
    print(f"  ✓ ra.repo._connect() returns {type(test_conn).__name__}")
    test_conn.close()
except Exception as e:
    print(f"  ✗ ra.repo._connect() raised: {e}")

print("\nChecking ra.repo.db_path ...")
try:
    print(f"  ✓ ra.repo.db_path = '{ra.repo.db_path}'")
except AttributeError:
    print("  ✗ ra.repo.db_path does NOT exist")

print("\n" + "="*60)
print("FIX #2 in _load_feature_data — change:")
print("    conn = ra.repo._conn")
print("to:")
print("    conn = ra.repo._connect()")
print("or:")
print("    conn = sqlite3.connect(ra.repo.db_path)")

Checking ra.repo._conn ...
  ✗ ra.repo._conn does NOT exist — secondary bug confirmed

Checking ra.repo._connect() ...
  ✓ ra.repo._connect() returns _GeneratorContextManager
  ✗ ra.repo._connect() raised: '_GeneratorContextManager' object has no attribute 'close'

Checking ra.repo.db_path ...
  ✓ ra.repo.db_path = 'data/training_results/2026_05_09_02\results.db'

FIX #2 in _load_feature_data — change:
    conn = ra.repo._conn
to:
    conn = ra.repo._connect()
or:
    conn = sqlite3.connect(ra.repo.db_path)


---
## 12. Summary and fixes

### Bug 1 (primary): fallback condition is wrong

The fluent API `rs.features().top(20)` returns 20 features with `weight = 0.0`. It can't parse the nested `{"stage_1": {...}, "stage_2": {...}}` format, so it zeros everything out.

The visualization code in `_load_feature_data` checks:
```python
if not raw_pairs:      # ← False! The list has 20 zero-weight items
    <sql fallback>     # ← Never reached
```

Later, the zero-filter removes everything:
```python
raw_pairs = [(f, w) for f, w in raw_pairs if w > 0]   # → empty list
```

**Fix**: Change the fallback condition from:
```python
if not raw_pairs:
```
to:
```python
if not raw_pairs or all(w == 0 for _, w in raw_pairs):
```

### Bug 2 (secondary): `ra.repo._conn` doesn't exist

Even with the fixed condition, the SQL fallback crashes because `ResultsRepository` doesn't have a `_conn` attribute. It has `_connect()` (method) and `db_path` (str).

**Fix**: Change:
```python
conn = ra.repo._conn
```
to:
```python
conn = ra.repo._connect()
```

In [14]:
# Final validation: confirm both fixes together produce plottable features
print("=" * 60)
print("VALIDATION: both fixes applied — does it produce features?")
print("=" * 60)

# Simulate the fixed _load_feature_data
raw_pairs = []
try:
    feat_result = rs.features().top(20)
    feat_dict = feat_result.as_dict()
    if MODEL_SLUG in feat_dict:
        raw_pairs = feat_dict[MODEL_SLUG]
    else:
        raw_pairs = next(iter(feat_dict.values()), [])
except Exception:
    pass

# ── FIX 1: improved fallback condition ─────────────────────────────────────
if not raw_pairs or all(w == 0 for _, w in raw_pairs):
    # ── FIX 2: correct connection access ──────────────────────────────────
    if conn is not None:
        try:
            cursor = conn.execute(
                "SELECT weights_json FROM features "
                "WHERE experiment_id = ? AND phase = 'enhanced'",
                (exp_id,),
            )
            feat_row = cursor.fetchone()

            if not feat_row or not feat_row[0]:
                cursor = conn.execute(
                    "SELECT weights_json FROM features "
                    "WHERE experiment_id = ? ORDER BY id DESC LIMIT 1",
                    (exp_id,),
                )
                feat_row = cursor.fetchone()

            if feat_row and feat_row[0]:
                raw_pairs = _parse_multistage_weights(feat_row[0], 20)
        except Exception as e:
            print(f"SQL fallback error: {e}")
    else:
        print("No DB connection — cannot test SQL fallback.")

# Zero-filter + normalize (same as visualization code)
raw_pairs = [(f, w) for f, w in raw_pairs if w > 0]
if raw_pairs:
    max_w = max(w for _, w in raw_pairs)
    if max_w > 0:
        raw_pairs = [(f, w / max_w) for f, w in raw_pairs]

print(f"\nFinal feature count: {len(raw_pairs)}")
if raw_pairs:
    print("\nTop-20 features (normalized):")
    for f, w in raw_pairs[:20]:
        bar = "█" * int(w * 30)
        print(f"  {f:35s} {w:.4f}  {bar}")
    print(f"\n✓ Chart would render with {len(raw_pairs)} bars.")
else:
    print("\n✗ Still empty — check cells 5-6 for clues.")

VALIDATION: both fixes applied — does it produce features?

Final feature count: 20

Top-20 features (normalized):
  dtp_avg                             1.0000  ██████████████████████████████
  opening_balance_flag                0.7288  █████████████████████
  payment_ratio                       0.4101  ████████████
  dtp_wavg                            0.3059  █████████
  opening_balance                     0.2179  ██████
  expected_survival_time              0.1512  ████
  surv_prob_58                        0.1322  ███
  dtp_1                               0.1138  ███
  surv_prob_306                       0.0728  ██
  plan_type_risk_score                0.0619  █
  surv_prob_1                         0.0586  █
  surv_prob_118                       0.0554  █
  cum_hazard_1                        0.0528  █
  dtp_2_trend                         0.0522  █
  due_month                           0.0522  █
  surv_prob_30                        0.0508  █
  dtp_3                             